# Accurate parser check on a Colab T4

Starts the accurate parser (MinerU 4) on this runtime's GPU, the way the CUDA image does
(vision-language model served by vLLM), and runs the same checks the container tests run on
CPU. Every check prints PASS or FAIL, and the last cell fails the notebook if any check failed.

**Before you start:** Runtime > Change runtime type > T4 GPU. This does not test the Docker
image (Colab has no Docker daemon); it tests the Python stack the image runs: MinerU, vLLM,
the service and the pinned models, on the GPU the image targets. Installing vLLM and
downloading the models takes a while (tens of minutes), and a failure at the install step is
itself a useful result: it means this vLLM release does not install on this runtime.

Set `BRANCH` below to the branch or tag to test.

In [ ]:
BRANCH = "pr-b-accurate"  # the branch or tag to test; use "main" once this is merged
REPO = "https://github.com/daddal001/two-tier-document-parser.git"
TIER = "standard"        # what the service uses on CUDA: the vision-language model plus the small models

## 1. The runtime

In [ ]:
import base64, subprocess, sys

def sh(command, check=True):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if check and result.returncode:
        raise RuntimeError(f"{command}\n{result.stdout}\n{result.stderr}")
    return (result.stdout + result.stderr).strip()

gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)
print("GPU:", gpu or "NONE FOUND - change the runtime type to T4 GPU")
print("Notebook Python:", sys.version.split()[0], "(the service gets its own Python 3.12 below)")

## 2. Get the code and a Python 3.12 environment

In [ ]:
import os
WORK = "/content/work"
sh(f"rm -rf {WORK} && mkdir -p {WORK}")
print(sh(f"git clone --depth 1 --branch {BRANCH} --recurse-submodules --shallow-submodules {REPO} {WORK}/repo"))
sh("pip install -q uv pypdf")  # pypdf is imported by the checks, which run in this kernel
print(sh(f"uv venv --python 3.12 {WORK}/venv"))
PY = f"{WORK}/venv/bin/python"
print(sh(f"{PY} --version"))

# Use the vLLM release the CUDA image is built on, so the notebook tests what the image runs.
import re
dockerfile = open(f"{WORK}/repo/deploy/Dockerfile.accurate").read()
VLLM = re.search(r"^FROM vllm/vllm-openai:v([0-9.]+)", dockerfile, re.M).group(1)
print("vLLM release from deploy/Dockerfile.accurate:", VLLM)

## 3. Install MinerU, vLLM and the service

Uses the repository's pinned dependency limits (`deploy/constraints/mineru-compat.txt`). vLLM
brings its own torch build, which is the point: this is the combination the image ships.

In [ ]:
REPO_DIR = f"{WORK}/repo"
# vLLM supplies the native stack, as the Docker base does. Its dependencies are held at the versions the
# base image ships (vllm-base.txt), the ones vLLM was run with; then the same Python pins are applied.
print(sh(f"cd {REPO_DIR} && uv pip install --python {PY} -c deploy/constraints/vllm-base.txt vllm=={VLLM}")[-800:])
install = (
    f"cd {REPO_DIR} && uv pip install --python {PY} --no-deps "
    "-r deploy/constraints/accurate-gpu.txt "
    "-e './MinerU[torch]' -e '.[accurate,telemetry,telemetry-sdk]'"
)
print(sh(install)[-800:])
print(sh(f"{PY} -c \"import torch, vllm; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'vllm', vllm.__version__)\""))

# vLLM's torch is built for one CUDA release; the runtime's driver must support it, or the engine cannot start.
driver_cuda = re.search(r"CUDA Version: ([0-9]+)\.", sh("nvidia-smi", check=False))
torch_cuda = sh(f"{PY} -c \"import torch; print(torch.version.cuda)\"")
print("torch is built for CUDA", torch_cuda, "| the driver supports CUDA", driver_cuda.group(1) if driver_cuda else "unknown")
if driver_cuda and torch_cuda.split(".")[0].isdigit() and int(torch_cuda.split(".")[0]) > int(driver_cuda.group(1)):
    print("WARNING: this torch needs a newer driver than this runtime has, so vLLM will fail to start. "
          "The CUDA image needs a host whose driver supports CUDA", torch_cuda, "(or the -cu129 base image tag).")


## 4. Download the models (pinned to the commits recorded in the repository)

In [ ]:
env = {
    **os.environ,
    "MINERU_HOME": f"{WORK}/mineru",
    "MINERU_DOWNLOAD_MODELS": TIER,
    "MINERU_MODEL_SMALL_BACKEND": "torch",
    "MINERU_MODEL_VLM_ENGINE": "vllm",
    "MINERU_MODEL_SOURCE": "huggingface",
    "OTEL_SDK_DISABLED": "true",
}
done = subprocess.run([PY, "-m", "two_tier_parser.accurate.prepare_models"], cwd=REPO_DIR, env=env,
                      text=True, capture_output=True)
print((done.stdout + done.stderr)[-800:])
assert done.returncode == 0, "model preparation failed"

## 5. Start the service and wait for it

In [ ]:
import time, urllib.request, urllib.error

# The venv is not activated, so its bin directory (where ninja lives) must be put on PATH by hand.
# vLLM's FlashInfer sampler would compile a kernel on first use; the CUDA image ships those
# kernels prebuilt, a Colab venv does not, so use vLLM's plain PyTorch sampler here.
service_env = {**env, "MINERU_MODEL_SOURCE": "local", "WORKERS": "1", "MAX_INFLIGHT_DOCUMENTS": "1",
               "UVICORN_TIMEOUT_KEEP_ALIVE": "600", "VLLM_USE_FLASHINFER_SAMPLER": "0",
               "PATH": f"{WORK}/venv/bin:" + os.environ["PATH"]}
log = open(f"{WORK}/service.log", "w")
server = subprocess.Popen(
    [PY, "-m", "uvicorn", "two_tier_parser.accurate.main:app", "--host", "127.0.0.1", "--port", "8005"],
    cwd=REPO_DIR, env=service_env, stdout=log, stderr=subprocess.STDOUT)
BASE = "http://127.0.0.1:8005"
deadline = time.time() + 900  # loading the VLM takes a while
while True:
    try:
        if urllib.request.urlopen(BASE + "/health", timeout=3).status == 200:
            break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    assert server.poll() is None, "the service exited: " + open(f"{WORK}/service.log").read()[-1500:]
    assert time.time() < deadline, "the service never became healthy: " + open(f"{WORK}/service.log").read()[-1500:]
    time.sleep(2)
print("service is up")

## 6. The checks

In [ ]:
import io, json, re, socket, threading
from pypdf import PdfReader, PdfWriter

def two_pages(source_pdf):
    writer = PdfWriter()
    for page in PdfReader(source_pdf).pages[:2]:
        writer.add_page(page)
    out = io.BytesIO()
    writer.write(out)
    return out.getvalue()

def upload(base, path, pdf, name="doc.pdf", timeout=600):
    boundary = "colabcheck"
    body = (f'--{boundary}\r\nContent-Disposition: form-data; name="file"; filename="{name}"\r\n'
            "Content-Type: application/pdf\r\n\r\n").encode() + pdf + f"\r\n--{boundary}--\r\n".encode()
    request = urllib.request.Request(base + path, data=body, method="POST",
                                     headers={"Content-Type": f"multipart/form-data; boundary={boundary}"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.status, {k.lower(): v for k, v in response.headers.items()}, json.loads(response.read())
    except urllib.error.HTTPError as error:
        return error.code, {k.lower(): v for k, v in error.headers.items()}, json.loads(error.read() or "null")

def get_json(base, path):
    with urllib.request.urlopen(base + path, timeout=30) as response:
        return response.status, json.loads(response.read())

def run_checks(base, source_pdf, expect_gpu, log_text=None):
    """Run every check against a running accurate service; return {name: (passed, detail)}."""
    results = {}

    def check(name):
        def wrap(fn):
            try:
                detail = fn() or ""
                results[name] = (True, detail)
            except Exception as error:  # a failed check is a result, not a crash
                results[name] = (False, f"{type(error).__name__}: {error}"[:300])
            return fn
        return wrap

    pdf = two_pages(source_pdf)

    @check("health reports the expected device")
    def _():
        status, body = get_json(base, "/health")
        assert status == 200 and body["status"] == "healthy", body
        assert body["gpu_available"] is expect_gpu, f"gpu_available={body['gpu_available']}, expected {expect_gpu}"
        return str(body)

    @check("a two-page parse returns Markdown from MinerU 4")
    def _():
        status, _, body = upload(base, "/parse", pdf)
        assert status == 200, (status, body)
        assert body["markdown"].strip(), "empty Markdown"
        assert body["metadata"]["pages"] == 2, body["metadata"]
        assert body["metadata"]["version"].startswith("4."), body["metadata"]["version"]
        assert body["metadata"]["gpu_used"] is expect_gpu, f"gpu_used={body['metadata']['gpu_used']}"
        return f"{len(body['markdown'])} characters in {body['metadata']['processing_time_ms']} ms"

    @check("a second document is refused with 503 and Retry-After while one is parsing")
    def _():
        first = {}
        worker = threading.Thread(target=lambda: first.update(result=upload(base, "/parse", pdf)))
        worker.start()
        time.sleep(1.5)
        status, headers, body = upload(base, "/parse", pdf)
        health_status, _ = get_json(base, "/health")
        worker.join(600)
        assert status == 503 and headers.get("retry-after") == "5" and body == {"detail": "document_admission_saturated"}, (status, headers, body)
        assert health_status == 200, "health was not available during the parse"
        assert first["result"][0] == 200, first["result"][0]

    @check("an oversized declared upload is refused with 413 without being read")
    def _():
        host, port = base.split("//")[1].split(":")
        with socket.create_connection((host, int(port)), timeout=30) as connection:
            connection.sendall(b"POST /parse HTTP/1.1\r\nHost: x\r\nContent-Type: multipart/form-data; boundary=x\r\n"
                               b"Content-Length: " + str(600 * 1024 * 1024).encode() + b"\r\nConnection: close\r\n\r\n")
            line = connection.makefile("rb").readline()
        assert b" 413 " in line, line

    @check("every log line after startup is JSON")
    def _():
        if log_text is None:
            return "skipped (no log supplied)"
        lines = [l for l in log_text.splitlines() if l.strip()]
        started = next(i for i, l in enumerate(lines) if "Application startup" in l)
        native = re.compile(r"\[[A-Z]:onnxruntime:|^Object \d+ \d+ not defined")
        bad = [l[:120] for l in lines[started:] if not native.search(l) and not l.startswith("{")]
        assert not bad, bad[:3]

    return results

def report(results):
    width = max(len(name) for name in results)
    for name, (passed, detail) in results.items():
        print(("PASS" if passed else "FAIL"), name.ljust(width), "-", detail)
    failed = [name for name, (passed, _) in results.items() if not passed]
    print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
    return failed

def show_failure_log(log_path, results, tail=60):
    """Print the service log lines that explain a failure: those naming a correlation id, then the tail."""
    lines = open(log_path).read().splitlines()
    for cid in sorted(set(re.findall(r"correlation_id=([0-9a-f]+)", json.dumps(results)))):
        print(f"--- service log lines for correlation_id {cid} ---")
        for line in lines:
            if cid in line:
                print(line[:6000])
    print(f"--- last {tail} lines of the service log ---")
    print("\n".join(line[:1500] for line in lines[-tail:]))

def inline_images(markdown, images):
    """Replace the image_id references in the Markdown with data URIs, so a notebook can display them."""
    for image in images:
        markdown = markdown.replace(f"]({image['image_id']})", f"](data:image/png;base64,{image['image_base64']})")
    return markdown


In [ ]:
results = run_checks(BASE, f"{REPO_DIR}/examples/data/sample.pdf", expect_gpu=True,
                     log_text=open(f"{WORK}/service.log").read())
failed = report(results)
print("\nGPU memory in use:", sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader", check=False))
if failed:
    show_failure_log(f"{WORK}/service.log", results)


## 7. The parse result

The checks above use two pages. This section parses the whole sample document, the paper
`examples/data/sample.pdf` that the visualization notebook uses (Y. Huang, C.-H. Chen and C.-J. Huang,
"Motor Fault Detection and Feature Extraction Using RNN-Based Variational Autoencoder", *IEEE Access*
7, 2019, [10.1109/ACCESS.2019.2940769](https://doi.org/10.1109/ACCESS.2019.2940769), published under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)), and shows what the service returns:
metadata, Markdown with its pictures, tables and formulas. On a T4 this takes several minutes.

In [ ]:
SAMPLE = f"{REPO_DIR}/examples/data/sample.pdf"
started = time.time()
with open(SAMPLE, "rb") as f:
    status, _, result = upload(BASE, "/parse", f.read(), name="sample.pdf", timeout=3600)
elapsed = time.time() - started
if status != 200:
    show_failure_log(f"{WORK}/service.log", {"parse": (False, str(result))})
assert status == 200, (status, result)
print(f"Parsed the sample ({os.path.getsize(SAMPLE) / 1024:.0f} KB) in {elapsed:.0f} s")
json.dump(result, open(f"{WORK}/sample-result.json", "w"), ensure_ascii=False, indent=2)

### 7.1 Metadata

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image

meta = result["metadata"]
display(pd.DataFrame([meta]).T.rename(columns={0: "value"}))
print(f"{len(result['images'])} images, {len(result['tables'])} tables, {len(result['formulas'])} formulas, "
      f"{len(result['markdown']):,} characters of Markdown")

result_checks = {
    "the sample's 11 pages were parsed": meta["pages"] == 11,
    "the vision-language model ran on the GPU": meta["backend"] == "vlm" and meta["gpu_used"] is True,
    "the paper's title is in the Markdown": "Variational Autoencoder" in result["markdown"],
    "the Markdown refers to images by id instead of embedding them": "base64," not in result["markdown"],
    "images, tables and formulas were extracted": all(result[k] for k in ("images", "tables", "formulas")),
}
for name, passed in result_checks.items():
    print("PASS" if passed else "FAIL", name)

### 7.2 Markdown, with the pictures it refers to

In [ ]:
display(Markdown(inline_images(result["markdown"], result["images"])))

### 7.3 Extracted images

In [ ]:
for number, image in enumerate(result["images"], 1):
    print(f"Image {number}/{len(result['images'])}: {image['image_id']}, page {image['page']}, bbox {image.get('bbox')}")
    display(Image(data=base64.b64decode(image["image_base64"])))

### 7.4 Extracted tables

In [ ]:
for number, table in enumerate(result["tables"], 1):
    print(f"Table {number}/{len(result['tables'])}: {table['table_id']}, page {table['page']}, bbox {table.get('bbox')}")
    display(Markdown(table["markdown"]))

### 7.5 Extracted formulas

In [ ]:
for number, formula in enumerate(result["formulas"], 1):
    print(f"Formula {number}/{len(result['formulas'])}: {formula['formula_id']}, page {formula['page']}")
    display(Markdown("$$" + formula["latex"] + "$$"))

## 8. Result

In [ ]:
server.terminate()
failed_results = [name for name, passed in result_checks.items() if not passed]
assert not failed and not failed_results, f"failed checks: {failed}; failed result checks: {failed_results}"
print("All checks passed on", gpu, "with branch", BRANCH)